In [8]:
tabla='cmdia10'
dim='dim_cie10'

In [9]:
import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [10]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM {tabla}", con=connection2)

connection2.close()

In [11]:
base2

,diagcod,diagdes,diagraiz,diagniv,diagsexcob,indplancapcobcod,diagenftra,diagregtipo,diagacptflg,estregcod,edxcapcod,edxgpocod,edxsgpcod,diaggpcflg,diagenfrara,diagescglaflg,diagpeasflg,diagdengflg
0,A63.0,VERRUGAS (VENEREAS) ANOGENITALES,01,0500,2,3,0,1,1,1,01,05,00,NaN,NaN,NaN,NaN,NaN
1,A63.8,OTRAS ENFERMEDADES DE TRANSMISION PREDOMINANTE...,01,0500,2,3,0,1,1,1,01,05,00,NaN,NaN,NaN,NaN,NaN
2,A64,ENFERMEDAD DE TRANSMISION SEXUAL NO ESPECIFICADA,01,0500,2,3,0,0,1,1,01,05,00,NaN,NaN,NaN,NaN,NaN
3,A65,SIFILIS NO VENEREA,01,0600,2,3,0,0,1,1,01,06,00,NaN,NaN,NaN,NaN,NaN
4,A66,FRAMBESIA,01,0600,2,3,0,0,0,1,01,06,00,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14320,I67.8,OTRAS ENFERMEDADES CEREBROVASCULARES ESPECIFIC...,09,0700,2,3,0,1,1,1,09,07,00,NaN,0.0,1.0,NaN,NaN
14321,D33.0,"TUMOR BENIGNO DEL ENCEFALO, SUPRATENTORIAL",02,0300,2,3,0,1,1,1,02,03,00,NaN,1.0,1.0,NaN,NaN
14322,Q93.6,SUPRESIONES VISIBLES SOLO EN LA PROMETAFASE,17,1100,2,3,0,1,1,1,17,11,00,NaN,1.0,NaN,NaN,NaN
14323,Q06.8,OTRAS MALFORMACIONES CONGENITAS ESPECIFICADAS ...,17,0100,2,3,0,1,1,1,17,01,00,NaN,1.0,NaN,NaN,NaN


In [12]:
base2.columns

Index(['diagcod', 'diagdes', 'diagraiz', 'diagniv', 'diagsexcob',
       'indplancapcobcod', 'diagenftra', 'diagregtipo', 'diagacptflg',
       'estregcod', 'edxcapcod', 'edxgpocod', 'edxsgpcod', 'diaggpcflg',
       'diagenfrara', 'diagescglaflg', 'diagpeasflg', 'diagdengflg'],
      dtype='object')

In [13]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


query_count_before = f"SELECT COUNT(*) FROM {tabla}"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} antes de la inserción: {cant_antes}")


#connection3.execute('CREATE TEMPORARY TABLE tmp_cbtci10 ()')
base2.to_sql(name=f'tmp_{tabla}', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cbtci10 FALSO CON LO OBTENIDO DEL ORACLE

query=f"""

ALTER TABLE tmp_{tabla}
ALTER COLUMN diagcod  TYPE character(8),
ALTER COLUMN diagdes  TYPE character(250);

UPDATE {tabla}
SET 
diagcod = t.diagcod,
diagdes = t.diagdes

FROM tmp_{tabla} t 
WHERE {tabla}.diagcod = t.diagcod AND TRIM({tabla}.diagcod) <> '' AND {tabla}.diagcod IS NOT NULL;

INSERT INTO {tabla}
(diagcod,diagdes) 
SELECT 
diagcod,diagdes

FROM tmp_{tabla}  t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {tabla} 
    WHERE {tabla}.diagcod = t.diagcod and {tabla}.diagcod IS NOT NULL
) ;


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
SELECT COUNT(*) FROM {tabla};
"""


c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")
base2 = pd.read_sql_query(f"SELECT * FROM {tabla}", con=connection3)


connection3.close()


Cantidad de filas en la tabla cmdia10 antes de la inserción: 14325
Cantidad de filas en la tabla cmdia10 después de la inserción: 14325
La cantidad de filas insertadas fue de: 0


In [14]:
#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT
base2.to_sql(name=f'tmp_{tabla}', con=engine4, if_exists='replace', index=False)

#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS
query_count_before = f"SELECT COUNT(*) FROM {dim}"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} antes de la inserción: {cant_antes}")

query=f"""

ALTER TABLE tmp_{tabla} 
ALTER COLUMN diagcod  TYPE character(8),
ALTER COLUMN diagdes  TYPE character(250);


INSERT INTO {dim} 

(cod_diag,des_diag) 

SELECT 

diagcod,diagdes

FROM tmp_{tabla} t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {dim} d 
    WHERE 
    
    d.cod_diag = t.diagcod AND
    d.des_diag = t.diagdes 
);
"""



c1= text(query)
cursor=connection4.execute(c1)

#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
SELECT COUNT(*) FROM {dim};
"""

c2= text(query2)
cursor=connection4.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla dim_cie10 antes de la inserción: 14325
Cantidad de filas en la tabla dim_cie10 después de la inserción: 14325
La cantidad de filas insertadas fue de: 0
